# EE348 - Lab 4: First-Order Logic Representation and Model Checking in Python

**Muhammad Sikandar Hussain 502808** 

This notebook completes Tasks 1-8 and the Post-Lab Questions. Each task gives the logical statement, its FOL representation, executable Python, output, and a short interpretation.

## Notation used

- $\forall x$: for every object $x$
- $\exists x$: there exists an object $x$
- $\land$: and
- $\lor$: or
- $\neg$: not
- $\rightarrow$: implies

## Task 1 - Constructing a First-Order Logic Interpretation

The interpretation contains a finite domain, subsets for object types, and relations represented by sets of tuples. A unary predicate takes one object; a binary predicate takes two.

In [1]:
# University domain and interpretation
domain = {"Ali", "Sara", "Ahmed", "KRR", "AI", "DS", "SEECS"}
students = {"Ali", "Sara"}
teachers = {"Ahmed"}
courses = {"KRR", "AI", "DS"}
universities = {"SEECS"}

enrolled = {("Ali", "KRR"), ("Ali", "AI"), ("Sara", "DS")}
teaches = {("Ahmed", "KRR"), ("Ahmed", "AI")}

def Student(x):
    return x in students

def Teacher(x):
    return x in teachers

def Course(x):
    return x in courses

def University(x):
    return x in universities

def Enrolled(x, y):
    return (x, y) in enrolled

def Teaches(x, y):
    return (x, y) in teaches

tests = [
    ('Student("Ali")', Student("Ali")),
    ('Student("Ahmed")', Student("Ahmed")),
    ('Course("KRR")', Course("KRR")),
    ('Course("Sara")', Course("Sara")),
    ('Enrolled("Ali", "KRR")', Enrolled("Ali", "KRR")),
    ('Enrolled("Sara", "KRR")', Enrolled("Sara", "KRR")),
    ('Teaches("Ahmed", "AI")', Teaches("Ahmed", "AI")),
    ('Teaches("Ahmed", "DS")', Teaches("Ahmed", "DS")),
]
for expression, value in tests:
    print(f"{expression:<30} -> {value}")

Student("Ali")                 -> True
Student("Ahmed")               -> False
Course("KRR")                  -> True
Course("Sara")                 -> False
Enrolled("Ali", "KRR")         -> True
Enrolled("Sara", "KRR")        -> False
Teaches("Ahmed", "AI")         -> True
Teaches("Ahmed", "DS")         -> False


### Classification and explanation

| Symbol | Classification | Reason |
|---|---|---|
| `Ali` | Constant (and the name of a domain object) | It denotes one particular individual in the interpretation. |
| `x` | Variable | It can range over objects in the domain. |
| `Student` | Unary predicate | It maps one object to `True` or `False`. |
| `Enrolled` | Binary predicate | It states a relation between a student and a course. |
| `KRR` | Constant (and the name of a domain object) | It denotes one particular course. |
| `Teaches` | Binary predicate | It states a relation between a teacher and a course. |
| `y` | Variable | It can range over objects in the domain. |

$Student(Ali)$ is a **closed atomic formula** about the particular object named `Ali`, so its truth value can be evaluated immediately. $Student(x)$ contains the free variable $x$; it becomes a definite proposition only after assigning an object to $x$ or binding $x$ with a quantifier.

The output confirms which named objects satisfy the unary predicates and which ordered pairs belong to the binary relations.

## Task 2 - Universal and Existential Quantification

### Part A - Universal quantifier

1. **English:** Every student is a member of the set of students.  
   **FOL:** $\forall x\,(Student(x) \rightarrow x \in students)$

2. **English:** Every student is enrolled in KRR.  
   **FOL:** $\forall x\,(Student(x) \rightarrow Enrolled(x,KRR))$

In [2]:
every_student_in_students = all(
    (not Student(x)) or (x in students) for x in domain
)
every_student_in_krr = all(
    (not Student(x)) or Enrolled(x, "KRR") for x in domain
)
krr_counterexamples = sorted(
    x for x in domain if Student(x) and not Enrolled(x, "KRR")
)

print("Every student belongs to students:", every_student_in_students)
print("Every student is enrolled in KRR:", every_student_in_krr)
print("Counterexample(s):", krr_counterexamples)

Every student belongs to students: True
Every student is enrolled in KRR: False
Counterexample(s): ['Sara']


The first result is `True` because `Student(x)` was defined as membership in `students`. The second is `False`: Sara is a student but `(Sara, KRR)` is not in the `enrolled` relation, so **Sara is the counterexample**. Non-students do not invalidate an implication whose antecedent is false.

### Part B - Existential quantifier

1. **English:** Some student is enrolled in KRR.  
   **FOL:** $\exists x\,(Student(x) \land Enrolled(x,KRR))$

2. **English:** Some student is enrolled in DS.  
   **FOL:** $\exists x\,(Student(x) \land Enrolled(x,DS))$

In [3]:
some_student_in_krr = any(
    Student(x) and Enrolled(x, "KRR") for x in domain
)
some_student_in_ds = any(
    Student(x) and Enrolled(x, "DS") for x in domain
)
ds_witnesses = sorted(
    x for x in domain if Student(x) and Enrolled(x, "DS")
)

print("Some student is enrolled in KRR:", some_student_in_krr)
print("Some student is enrolled in DS:", some_student_in_ds)
print("DS witness(es):", ds_witnesses)

Some student is enrolled in KRR: True
Some student is enrolled in DS: True
DS witness(es): ['Sara']


Both existential statements are `True`. Ali witnesses the KRR statement, while Sara witnesses the DS statement; the displayed DS witness list makes the second result explicit.

## Task 3 - Nested Quantifiers

1. **English:** Every student is enrolled in at least one course.  
   **FOL:** $\forall x\,(Student(x) \rightarrow \exists y\,(Course(y) \land Enrolled(x,y)))$

2. **Exercise 3.1:** Every teacher teaches at least one course.  
   **FOL:** $\forall x\,(Teacher(x) \rightarrow \exists y\,(Course(y) \land Teaches(x,y)))$

3. **Exercise 3.2:** Every course is taught by at least one teacher.  
   **FOL:** $\forall y\,(Course(y) \rightarrow \exists x\,(Teacher(x) \land Teaches(x,y)))$

**Prediction for Exercise 3.2:** `False`, because DS has no teacher in the original `teaches` relation.

In [4]:
every_student_has_course = all(
    (not Student(x))
    or any(Course(y) and Enrolled(x, y) for y in domain)
    for x in domain
)

every_teacher_has_course = all(
    (not Teacher(x))
    or any(Course(y) and Teaches(x, y) for y in domain)
    for x in domain
)

every_course_has_teacher = all(
    (not Course(y))
    or any(Teacher(x) and Teaches(x, y) for x in domain)
    for y in domain
)

untaught_courses = sorted(
    y for y in domain
    if Course(y) and not any(Teacher(x) and Teaches(x, y) for x in domain)
)

print("Every student has at least one course:", every_student_has_course)
print("Every teacher teaches at least one course:", every_teacher_has_course)
print("Every course has at least one teacher:", every_course_has_teacher)
print("Untaught course(s):", untaught_courses)

Every student has at least one course: True
Every teacher teaches at least one course: True
Every course has at least one teacher: False
Untaught course(s): ['DS']


The first statement is `True`: Ali has KRR and AI, and Sara has DS. The second is `True`: the only teacher, Ahmed, teaches KRR and AI. The prediction for the third is confirmed: it is `False` because **DS is the counterexample**.

## Task 4 - Quantifier Order

### Statement A

**English:** Every student is enrolled in some course.  
**FOL:** $\forall x\,(Student(x) \rightarrow \exists y\,(Course(y) \land Enrolled(x,y)))$

### Statement B

**English:** There is a course in which every student is enrolled.  
**FOL:** $\exists y\,(Course(y) \land \forall x\,(Student(x) \rightarrow Enrolled(x,y)))$

In [5]:
A = all(
    (not Student(x))
    or any(Course(y) and Enrolled(x, y) for y in domain)
    for x in domain
)

B = any(
    Course(y)
    and all((not Student(x)) or Enrolled(x, y) for x in domain)
    for y in domain
)

print("A - every student has some course:", A)
print("B - one course has every student:", B)

A - every student has some course: True
B - one course has every student: False


$\forall x\exists y$ permits the choice of $y$ to depend on $x$: Ali can use KRR (or AI), while Sara can use DS. By contrast, $\exists y\forall x$ requires one fixed course $y$ that works for every student. No course contains both Ali and Sara, so A is `True` and B is `False`. Therefore, “every student studies some course” does not imply that all students study the same course.

## Task 5 - Functions in FOL

A predicate returns a truth value. A function denotes/returns an object (or `None` here when the finite interpretation has not assigned an instructor).

In [6]:
instructor_of = {"KRR": "Ahmed", "AI": "Ahmed", "DS": None}

def InstructorOf(course):
    return instructor_of.get(course)

print('InstructorOf("KRR"):', InstructorOf("KRR"))
print('InstructorOf("AI"):', InstructorOf("AI"))
print('InstructorOf("DS"):', InstructorOf("DS"))
print('Teacher("Ahmed"):', Teacher("Ahmed"))
print('InstructorOf("KRR"):', InstructorOf("KRR"))
print('Type from Teacher:', type(Teacher("Ahmed")).__name__)
print('Type from InstructorOf:', type(InstructorOf("KRR")).__name__)

InstructorOf("KRR"): Ahmed
InstructorOf("AI"): Ahmed
InstructorOf("DS"): None
Teacher("Ahmed"): True
InstructorOf("KRR"): Ahmed
Type from Teacher: bool
Type from InstructorOf: str


`Teacher("Ahmed")` returns a Boolean (`bool`) and expresses the proposition $Teacher(Ahmed)$. `InstructorOf("KRR")` returns the domain object name `Ahmed` (represented by a Python `str`) and expresses the function term $InstructorOf(KRR)=Ahmed$. Thus, predicates are truth-valued, whereas functions return objects.

### Exercise 5.1

After adding Ahmed as the instructor for DS, evaluate:

**English:** Every course has an instructor.  
**FOL (using a definedness test for this Python model):** $\forall c\,(Course(c) \rightarrow InstructorOf(c) \neq None)$

In [7]:
instructor_of["DS"] = "Ahmed"
print('InstructorOf("DS") after update:', InstructorOf("DS"))

every_course_has_instructor = all(
    (not Course(c)) or (InstructorOf(c) is not None) for c in domain
)
print("Every course has an instructor:", every_course_has_instructor)

InstructorOf("DS") after update: Ahmed
Every course has an instructor: True


The result is `True` under the updated interpretation because KRR, AI, and DS all map to Ahmed. In standard FOL functions are normally total; the initial use of `None` models a temporarily undefined entry in Python, and the update makes the function defined for every course.

## Task 6 - Witnesses and Counterexamples

- A **witness** is an object that satisfies the body of a true existential statement.
- A **counterexample** is an object that violates the body of a claimed universal statement.

In [8]:
print("Witnesses for: some student is enrolled in KRR")
for x in sorted(domain):
    if Student(x) and Enrolled(x, "KRR"):
        print("Witness:", x)

print("\nCounterexamples for: every student is enrolled in KRR")
for x in sorted(domain):
    if Student(x) and not Enrolled(x, "KRR"):
        print("Counterexample:", x)

print("\nExercise 6.1 - witness for a course taught by Ahmed")
for c in sorted(domain):
    if Course(c) and Teaches("Ahmed", c):
        print("Witness course:", c)
        break

print("\nExercise 6.2 - counterexample to every course being taught by Ahmed")
for c in sorted(domain):
    if Course(c) and not Teaches("Ahmed", c):
        print("Counterexample course:", c)

Witnesses for: some student is enrolled in KRR
Witness: Ali

Counterexamples for: every student is enrolled in KRR
Counterexample: Sara

Exercise 6.1 - witness for a course taught by Ahmed
Witness course: AI

Exercise 6.2 - counterexample to every course being taught by Ahmed
Counterexample course: DS


Ali witnesses $\exists x(Student(x)\land Enrolled(x,KRR))$. Sara is a counterexample to $\forall x(Student(x)\rightarrow Enrolled(x,KRR))$. For Exercise 6.1, the loop prints one valid witness course taught by Ahmed. For Exercise 6.2, DS is the required counterexample because Ahmed does not teach it in the `teaches` relation. (Updating `InstructorOf(DS)` in Task 5 does not alter `Teaches(Ahmed,DS)`.)

## Task 7 - English to FOL to Python

The existing university interpretation is used below. Each item identifies its non-logical symbols, presents the FOL formula, evaluates the corresponding Python expression, and is explained after the output.

| # | English statement | Predicates/constants | FOL formula |
|---|---|---|---|
| 1 | Ali is a student. | `Student`; `Ali` | $Student(Ali)$ |
| 2 | Some student studies KRR. | `Student`, `Enrolled`; `KRR` | $\exists x(Student(x)\land Enrolled(x,KRR))$ |
| 3 | Every student is enrolled in at least one course. | `Student`, `Course`, `Enrolled` | $\forall x(Student(x)\rightarrow\exists y(Course(y)\land Enrolled(x,y)))$ |
| 4 | Every teacher teaches at least one course. | `Teacher`, `Course`, `Teaches` | $\forall x(Teacher(x)\rightarrow\exists y(Course(y)\land Teaches(x,y)))$ |
| 5 | There is a teacher who teaches KRR. | `Teacher`, `Teaches`; `KRR` | $\exists x(Teacher(x)\land Teaches(x,KRR))$ |
| 6 | There is a teacher who teaches every course. | `Teacher`, `Course`, `Teaches` | $\exists x(Teacher(x)\land\forall y(Course(y)\rightarrow Teaches(x,y)))$ |
| 7 | No student is a teacher. | `Student`, `Teacher` | $\forall x(Student(x)\rightarrow\neg Teacher(x))$ |
| 8 | Some student is enrolled in every course. | `Student`, `Course`, `Enrolled` | $\exists x(Student(x)\land\forall y(Course(y)\rightarrow Enrolled(x,y)))$ |

In [9]:
task7_results = {
    "1. Student(Ali)": Student("Ali"),
    "2. Some student studies KRR": any(
        Student(x) and Enrolled(x, "KRR") for x in domain
    ),
    "3. Every student has at least one course": all(
        (not Student(x))
        or any(Course(y) and Enrolled(x, y) for y in domain)
        for x in domain
    ),
    "4. Every teacher teaches at least one course": all(
        (not Teacher(x))
        or any(Course(y) and Teaches(x, y) for y in domain)
        for x in domain
    ),
    "5. A teacher teaches KRR": any(
        Teacher(x) and Teaches(x, "KRR") for x in domain
    ),
    "6. A teacher teaches every course": any(
        Teacher(x)
        and all((not Course(y)) or Teaches(x, y) for y in domain)
        for x in domain
    ),
    "7. No student is a teacher": all(
        (not Student(x)) or (not Teacher(x)) for x in domain
    ),
    "8. A student is enrolled in every course": any(
        Student(x)
        and all((not Course(y)) or Enrolled(x, y) for y in domain)
        for x in domain
    ),
}

for statement, result in task7_results.items():
    print(f"{statement:<48} -> {result}")

1. Student(Ali)                                  -> True
2. Some student studies KRR                      -> True
3. Every student has at least one course         -> True
4. Every teacher teaches at least one course     -> True
5. A teacher teaches KRR                         -> True
6. A teacher teaches every course                -> False
7. No student is a teacher                       -> True
8. A student is enrolled in every course         -> False


### Task 7 result explanations

1. `True`: Ali belongs to the `students` set.
2. `True`: Ali is a student and is enrolled in KRR.
3. `True`: Ali has KRR/AI, and Sara has DS.
4. `True`: Ahmed is the only teacher and teaches KRR and AI.
5. `True`: Ahmed witnesses the existential statement.
6. `False`: Ahmed does not teach DS, and there is no other teacher.
7. `True`: the student set `{Ali, Sara}` and teacher set `{Ahmed}` are disjoint.
8. `False`: Ali lacks DS, while Sara lacks KRR and AI, so no student is enrolled in all three courses.

## Task 8 - Mini Model-Checker: Hospital Domain

This separate interpretation contains two patients, two doctors, two departments, and seven binary-relation facts (four `Treats` facts and three `WorksIn` facts).

In [10]:
# Hospital knowledge base
hospital_domain = {
    "Patient1", "Patient2", "DrAhmed", "DrSara", "Cardiology", "Neurology"
}
patients = {"Patient1", "Patient2"}
doctors = {"DrAhmed", "DrSara"}
departments = {"Cardiology", "Neurology"}

treats = {
    ("DrAhmed", "Patient1"),
    ("DrAhmed", "Patient2"),
    ("DrSara", "Patient1"),
    ("DrSara", "Patient2"),
}
works_in = {
    ("DrAhmed", "Cardiology"),
    ("DrSara", "Neurology"),
    ("DrSara", "Cardiology"),
}

def Patient(x):
    return x in patients

def Doctor(x):
    return x in doctors

def Department(x):
    return x in departments

def Treats(x, y):
    return (x, y) in treats

def WorksIn(x, y):
    return (x, y) in works_in

print("Hospital domain:", sorted(hospital_domain))
print("Treats facts:", sorted(treats))
print("WorksIn facts:", sorted(works_in))

Hospital domain: ['Cardiology', 'DrAhmed', 'DrSara', 'Neurology', 'Patient1', 'Patient2']
Treats facts: [('DrAhmed', 'Patient1'), ('DrAhmed', 'Patient2'), ('DrSara', 'Patient1'), ('DrSara', 'Patient2')]
WorksIn facts: [('DrAhmed', 'Cardiology'), ('DrSara', 'Cardiology'), ('DrSara', 'Neurology')]


### A. Every doctor works in at least one department

**FOL:** $\forall x(Doctor(x)\rightarrow\exists y(Department(y)\land WorksIn(x,y)))$

In [11]:
A_hospital = all(
    (not Doctor(x))
    or any(Department(y) and WorksIn(x, y) for y in hospital_domain)
    for x in hospital_domain
)
print(A_hospital)

True


`True`: DrAhmed works in Cardiology, and DrSara works in Neurology and Cardiology.

### B. Some doctor treats Patient1

**FOL:** $\exists x(Doctor(x)\land Treats(x,Patient1))$

In [12]:
B_hospital = any(
    Doctor(x) and Treats(x, "Patient1") for x in hospital_domain
)
B_witnesses = sorted(
    x for x in hospital_domain if Doctor(x) and Treats(x, "Patient1")
)
print(B_hospital)
print("Witness doctor(s):", B_witnesses)

True
Witness doctor(s): ['DrAhmed', 'DrSara']


`True`: DrAhmed and DrSara both witness the existential statement.

### C. Every patient is treated by at least one doctor

**FOL:** $\forall y(Patient(y)\rightarrow\exists x(Doctor(x)\land Treats(x,y)))$

In [13]:
C_hospital = all(
    (not Patient(y))
    or any(Doctor(x) and Treats(x, y) for x in hospital_domain)
    for y in hospital_domain
)
print(C_hospital)

True


`True`: each of Patient1 and Patient2 has at least one treating doctor.

### D. There is a doctor who treats every patient

**FOL:** $\exists x(Doctor(x)\land\forall y(Patient(y)\rightarrow Treats(x,y)))$

In [14]:
D_hospital = any(
    Doctor(x)
    and all((not Patient(y)) or Treats(x, y) for y in hospital_domain)
    for x in hospital_domain
)
D_witnesses = sorted(
    x for x in hospital_domain
    if Doctor(x)
    and all((not Patient(y)) or Treats(x, y) for y in hospital_domain)
)
print(D_hospital)
print("Witness doctor(s):", D_witnesses)

True
Witness doctor(s): ['DrAhmed', 'DrSara']


`True`: both DrAhmed and DrSara treat every patient in this finite interpretation.

### E. Additional quantified statement

**English:** Some department has at least two doctors working in it.  
**FOL:** $\exists d(Department(d)\land\exists x\exists y(Doctor(x)\land Doctor(y)\land x\neq y\land WorksIn(x,d)\land WorksIn(y,d)))$

In [15]:
E_hospital = any(
    Department(d)
    and any(
        Doctor(x)
        and any(
            Doctor(y)
            and x != y
            and WorksIn(x, d)
            and WorksIn(y, d)
            for y in hospital_domain
        )
        for x in hospital_domain
    )
    for d in hospital_domain
)
E_witnesses = sorted(
    d for d in hospital_domain
    if Department(d)
    and len([x for x in hospital_domain if Doctor(x) and WorksIn(x, d)]) >= 2
)
print(E_hospital)
print("Witness department(s):", E_witnesses)

True
Witness department(s): ['Cardiology']


`True`: Cardiology is a witness because both DrAhmed and DrSara work there.

## Post-Lab Questions

### 1. What is the difference between a constant and a variable in FOL?

A **constant** denotes one fixed object in an interpretation, such as `Ali` or `KRR`. A **variable**, such as $x$, may range over objects in the domain and receives a value through an assignment or quantifier.

### 2. What is the difference between a predicate and a function?

A **predicate** maps object(s) to a truth value, for example $Teacher(Ahmed)$. A **function** maps object(s) to another object, for example $InstructorOf(KRR)=Ahmed$.

### 3. What is the difference between $\forall x\,P(x)$ and $\exists x\,P(x)$?

$\forall x\,P(x)$ requires $P$ to hold for every object in the domain. $\exists x\,P(x)$ requires it to hold for at least one object.

### 4. Why is Python's `all()` a useful computational analogy for $\forall$ over a finite domain?

`all(P(x) for x in domain)` returns `True` exactly when every generated Boolean value is true, mirroring the requirement of a universal quantifier over each object of a finite domain.

### 5. Why is Python's `any()` a useful computational analogy for $\exists$?

`any(P(x) for x in domain)` returns `True` when at least one generated Boolean value is true, so a successful object acts as an existential witness.

### 6. Why is $\forall x\exists y\,R(x,y)$ generally different from $\exists y\forall x\,R(x,y)$?

In the first formula, a different $y$ may be chosen for each $x$. In the second, one fixed $y$ must relate to every $x$, which is normally a stronger requirement.

### 7. What is a witness for an existential statement?

A witness is a particular domain object (or tuple of objects) that makes the body of an existentially quantified formula true.

### 8. What is a counterexample to a universal statement?

A counterexample is a domain object for which the universally claimed property is false; one counterexample is enough to falsify the universal statement.

### 9. If `all(Student(x) for x in domain)` is `False`, what does this tell us?

Assuming the domain is non-empty, it tells us that at least one domain object is not a student. That object is a counterexample to $\forall x\,Student(x)$. It does not imply that no objects are students.

## Conclusion

This notebook constructed two finite interpretations and used `all()` and `any()` to model universal and existential quantification. It also demonstrated nested quantifiers, quantifier order, functions versus predicates, and the extraction of witnesses and counterexamples.